In [1]:
# IMPORTS
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from xgboost import XGBClassifier
# LOAD DATA
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")

test_ids = test["PassengerId"]

full = pd.concat([train, test], sort=False)
# ADVANCED FEATURE ENGINEERING

# Fill base missing values
full["Fare"] = full["Fare"].fillna(full["Fare"].median())
full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])
full["Cabin"] = full["Cabin"].fillna("U")

# Extract Title
full["Title"] = full["Name"].str.extract(" ([A-Za-z]+).", expand=False)

rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
full["Title"] = full["Title"].replace(rare_titles, "Rare")
full["Title"] = full["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})
full["Title"] = full["Title"].fillna("Rare")

# Family features
full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
full["IsAlone"] = (full["FamilySize"] == 1).astype(int)

# Cabin Deck
full["Deck"] = full["Cabin"].str[0]

# Age imputation by Title
full["Age"] = full.groupby("Title")["Age"].transform(lambda x: x.fillna(x.median()))
full["Age"] = full["Age"].fillna(full["Age"].median())

# Fare per person
full["FarePerPerson"] = full["Fare"] / full["FamilySize"]

# Age & Fare bins
full["AgeBin"] = pd.cut(full["Age"], 5, labels=False)
full["FareBin"] = pd.qcut(full["Fare"], 4, labels=False)

# Drop unused
full.drop(["PassengerId","Name","Ticket","Cabin"], axis=1, inplace=True)

# One-hot encoding
full = pd.get_dummies(full)

# Safety
full = full.fillna(0)
# SPLIT BACK
train_processed = full.iloc[:len(train)]
test_processed = full.iloc[len(train):]

X = train_processed.drop("Survived", axis=1)
y = train_processed["Survived"].astype(int)
X_test = test_processed.drop("Survived", axis=1)
# CROSS VALIDATION SETUP
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# BASE MODELS

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=7,
    min_samples_split=3,
    random_state=42
)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

svm = make_pipeline(
    StandardScaler(),
    SVC(C=10, gamma=0.01, kernel="rbf", probability=True)
)

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

# STACKING (4 MODELS)
stack = StackingClassifier(
    estimators=[
        ("rf", rf),
        ("gb", gb),
        ("svm", svm),
        ("xgb", xgb)
    ],
    final_estimator=LogisticRegression(),
    cv=5
)
# CROSS VALIDATION SCORE
cv_score = cross_val_score(stack, X, y, cv=skf, scoring="accuracy")
print("CV Accuracy:", cv_score.mean())
# TRAIN FULL MODEL
stack.fit(X, y)
pred = stack.predict(X_test)
# SUBMISSION
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": pred.astype(int)
})
submission.to_csv("submission.csv", index=False)

CV Accuracy: 0.8473291067729584
